# Data for the figures

The pipeline saves its fits but not four things the figures need. This notebook computes them from the final point and stores them in `OUT/paper_figs_data/`:

1. **Relative reduced sensitivities** $\partial\ln y^*/\partial\ln\theta$ of every measured output at the final point, per condition, for the free parameters (Fig. 2a). Computed exactly (implicit function theorem with CasADi derivatives, `sensitivity.py`), the same sensitivities the Fisher information is built from, but *relative* (divided by the output). Relative sensitivities diverge as an output approaches zero, which is why the figure excludes entries below 1% of the output median.
2. **The literature-parameter model.** All 37 parameters fixed at the literature values (`Data/theta_init_sampled.csv`, clipped into the theta bounds), the inner problem over $x, u, b$ solved under the same bounds: its predicted outputs (`y_literature_inner.csv`) and its objective breakdown (the objective of the model with all parameters fixed).
3. **The fit after the structural phase.** The selection refits once after all structural fixes and goes on with the practical loop without saving that fit. It is rebuilt here from the stage-0 fit and the structural decisions of `reduction/fixing_log.csv` (the same refit as `selection.structural_step`): its in-sample predictions (`y_structural_fit.csv`) and objective (`structural_phase.json`).
4. **Metrics per stage** (literature, stage 0, after the structural phase, final, LOOCV and the LOOCV training-mean baseline) on identical definitions: RMSE and $R^2$ against the per-output mean of the in-sample data (`stage_metrics.csv`).

Parts 1 to 3 solve inner problems and run behind `HK_RUN=1` (a few minutes); part 4 is arithmetic on saved files and is recomputed every time, so **re-run the notebook (no `HK_RUN` needed) after `loocv.ipynb`** to refresh `stage_metrics.csv`.

In [1]:
import os
import sys
import logging
import warnings

HERE = os.getcwd()
REPO = os.path.dirname(HERE) if os.path.basename(HERE) == "notebooks" else HERE
for path in (os.path.join(REPO, "notebooks"), os.path.join(REPO, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

import settings  # noqa: F401  (pins BLAS / OpenMP to one thread before numpy is loaded)
from paths import resolve_out

OUT = resolve_out(os.environ.get("HK_OUT") or settings.METHOD["out_dir"])   # default: out_dir of settings/method.csv; relative paths are relative to the repo root
RUN = os.environ.get("HK_RUN", "0") == "1"

warnings.filterwarnings("ignore")
logging.getLogger("pyomo").setLevel(logging.ERROR)
print("REPO     :", REPO)
print("OUT      :", OUT)
print("RUN      :", RUN, "(False = load the saved results of OUT; True = compute and save into OUT)")

import json

import numpy as np
import pandas as pd

from settings import THETA_BOUNDS, METHOD
from model.parameters import ALL_PARAMS
from estimation.anchors import load_point, read_anchor, restore_anchor, check_bounds
from estimation.refit import refit_with_fixed
from sensitivity import reduced_sensitivity
from metrics import pred_metrics

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

FINAL = os.path.join(OUT, "final")
DATA_OUT = os.path.join(OUT, "paper_figs_data")

def to_wide(y_by_condition):
    """{condition: Series(output -> value)} -> DataFrame (conditions x outputs)."""
    return pd.DataFrame(y_by_condition).T

def mean_abs_ignoring_nan(values):
    return float(np.nanmean(np.abs(values)))

REPO     : /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733
OUT      : /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/results/flux_model
RUN      : False (False = load the saved results of OUT; True = compute and save into OUT)


## 1. Sensitivities and the literature model (`HK_RUN=1`)

`load_point` rebuilds the final point as an estimator without a solve; `reduced_sensitivity(..., relative=True)` solves the inner problem of each condition once and differentiates its optimality conditions, so the table has one row per (condition, output) and one column per free parameter. For the literature model only the objective breakdown and the unperturbed outputs `y0` are needed (one column, `kcat_f_4`, is requested to trigger the solves at the literature theta).

In [2]:
if RUN:
    check_bounds()
    os.makedirs(DATA_OUT, exist_ok=True)
    state = read_anchor(FINAL)
    est = load_point(state)
    print("final point loaded: obj %.5f, %d free" % (state["obj"], len(state["free"])))

    # ---- relative reduced sensitivities at the final point ----
    final_sensitivity = reduced_sensitivity(est, relative=True)
    blocks = []
    for condition, table in final_sensitivity["sensitivities"].items():
        block = table.copy()
        block["condition"] = condition
        block["output"] = block.index
        blocks.append(block)
    long = pd.concat(blocks, ignore_index=True)
    long.to_csv(os.path.join(DATA_OUT, "sens_rel_by_condition.csv"), index=False)
    mean_abs = long.drop(columns="condition").groupby("output").agg(mean_abs_ignoring_nan)
    mean_abs.to_csv(os.path.join(DATA_OUT, "sens_rel_mean_abs.csv"))
    to_wide(final_sensitivity["y0"]).to_csv(os.path.join(DATA_OUT, "y_final_inner.csv"))
    print("sensitivities: %d conditions, outputs %s" % (len(final_sensitivity["sensitivities"]), list(mean_abs.index)))

    # ---- the literature-parameter model ----
    literature = pd.read_csv(os.path.join(REPO, "Data", "theta_init_sampled.csv"), index_col=0)["value"]
    theta_lit = {}
    for p in ALL_PARAMS:
        theta_lit[p] = float(min(max(float(literature[p]), THETA_BOUNDS[p][0]), THETA_BOUNDS[p][1]))
    literature_model = reduced_sensitivity(est, theta=theta_lit, param_names=["kcat_f_4"], relative=True)
    to_wide(literature_model["y0"]).to_csv(os.path.join(DATA_OUT, "y_literature_inner.csv"))
    print("literature model: conditions without a solution %s" % literature_model["failed"])
    est.objective_breakdown(theta=theta_lit).to_csv(os.path.join(DATA_OUT, "objective_breakdown_literature.csv"))
else:
    print("RUN is False: sensitivities and the literature model are loaded from", DATA_OUT)

RUN is False: sensitivities and the literature model are loaded from /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/results/flux_model/paper_figs_data


## 2. The fit after the structural phase (`HK_RUN=1`)

`restore_anchor` rebuilds the stage-0 fit; `refit_with_fixed` holds the structural fixes at their stage-0 values and refits the other parameters from that point, as the selection does. `predict_fitted` gives the in-sample predictions of that fit.

In [3]:
if RUN:
    stage0_est, _stage0_state = restore_anchor(os.path.join(OUT, "stage0"))
    fixing_log = pd.read_csv(os.path.join(OUT, "reduction", "fixing_log.csv"))
    structural_fixes = fixing_log.loc[(fixing_log["stage"] == "structural") & (fixing_log["action"] == "fixed"), "param"].tolist()
    free_after_structural = [p for p in stage0_est.free_params if p not in structural_fixes]
    structural_est = refit_with_fixed(stage0_est, free_after_structural, print)
    structural_prediction = structural_est.predict_fitted()
    structural_prediction.predicted.to_csv(os.path.join(DATA_OUT, "y_structural_fit.csv"))
    structural_row = {"n_free": len(structural_est.free_params), "n_fixed": len(structural_fixes),
                      "obj": float(structural_est.obj_value), "n_pred_failed": int(structural_prediction.n_failed)}
    with open(os.path.join(DATA_OUT, "structural_phase.json"), "w") as fh:
        json.dump(structural_row, fh, indent=1)
    print("after the structural phase: %d free (%d fixed), obj %.5f (stage 0: %.5f)" % (
        structural_row["n_free"], structural_row["n_fixed"], structural_row["obj"], stage0_est.obj_value))
else:
    print("RUN is False: the fit after the structural phase is loaded from", DATA_OUT)

RUN is False: the fit after the structural phase is loaded from /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/results/flux_model/paper_figs_data


### Saved sensitivities, literature model and structural phase

In [4]:
def read_saved(name, **kw):
    path = os.path.join(DATA_OUT, name)
    if os.path.exists(path):
        return pd.read_csv(path, **kw)
    print("not found (computed with HK_RUN=1): paper_figs_data/" + name)
    return None

sens_mean = read_saved("sens_rel_mean_abs.csv", index_col=0)
if sens_mean is not None:
    print("mean |d ln y / d ln theta| over the conditions (rows: outputs, columns: free parameters)")
    display(sens_mean)

bd_lit = read_saved("objective_breakdown_literature.csv", index_col=0)
if bd_lit is not None:
    print("objective of the literature model (mean over conditions): %.2f" % bd_lit.loc["MEAN", "total"])

structural_path = os.path.join(DATA_OUT, "structural_phase.json")
structural_row = json.load(open(structural_path)) if os.path.exists(structural_path) else None
if structural_row is None:
    print("not found (computed with HK_RUN=1): paper_figs_data/structural_phase.json")
else:
    print("after the structural phase: %d free, obj %.5f" % (structural_row["n_free"], structural_row["obj"]))

mean |d ln y / d ln theta| over the conditions (rows: outputs, columns: free parameters)


,v_max_1,Ks_g6p_pgi,kcat_f_2,kcat_f_3,Ks_fbp_4,kcat_f_4,kcat_f_5,kcat_f_6,kcat_f_7,Ks_adp_7,kcat_f_8,Ks_3pg_8,kcat_f_9,Ks_2pg_9
output,,,,,,,,,,,,,,
C_3pg,1.051038e-07,2.495799e-05,1.082905e-03,0.324120,0.016510,0.300920,2.893307e-03,9.348940e-04,8.277836e-03,2.289928e-03,0.008689,0.004304,0.408764,0.014304
C_dhap,2.562998e-09,4.500056e-06,9.723548e-05,0.043018,0.003791,0.148239,1.080079e-04,4.533072e-05,2.507209e-04,4.197940e-05,0.000046,0.000058,0.005134,0.000299
C_f6p,2.717084e-10,9.211540e-05,2.776917e-03,0.043200,0.002475,0.018381,2.906537e-05,2.681639e-05,1.728420e-04,9.127399e-06,0.000063,0.000020,0.009120,0.000162
C_fbp,2.822764e-09,1.632375e-05,5.929432e-04,0.216030,0.030842,0.154709,8.892345e-05,9.912793e-05,7.929623e-04,4.827789e-05,0.000258,0.000081,0.034695,0.000646
C_g6p,6.095382e-05,1.767359e-04,5.858205e-03,0.041667,0.002952,0.018767,2.767170e-05,2.539980e-05,1.657839e-04,9.227717e-06,0.000194,0.000108,0.008609,0.000165
C_pep,1.211319e-07,1.773625e-05,7.109879e-04,0.270667,0.014395,0.387490,2.335514e-03,7.963000e-04,6.867654e-03,2.422214e-03,0.009473,0.006584,0.311807,0.031703
v_gapA,1.168523e-06,2.781952e-06,1.196535e-04,0.038163,0.005315,0.159568,8.002066e-06,2.458535e-05,1.999076e-04,1.341594e-05,0.000165,0.000034,0.019325,0.000231
v_gpmA,6.805918e-05,2.968882e-06,1.279763e-04,0.040738,0.005635,0.169701,9.704015e-06,2.412102e-05,1.945510e-04,1.052312e-05,0.000395,0.000160,0.021991,0.000364
v_pfkB,2.308176e-04,6.416759e-06,2.023308e-04,0.069808,0.018584,0.501710,2.251939e-05,4.312616e-05,3.096529e-04,1.783271e-05,0.000632,0.000369,0.017005,0.000293


objective of the literature model (mean over conditions): 7771.70
after the structural phase: 29 free, obj 10.90867


## 3. Metrics per stage

The in-sample data of the final analysis (`final/pred_insample_real.csv`) is the common reference. Predictions: the literature inner problem, the stage-0 in-sample predictions, the fit after the structural phase, the final in-sample predictions, the LOOCV held-out predictions, and the LOOCV training-mean baseline. Every $R^2$ uses the per-output mean of the in-sample data as the reference value; `n` and the RMSE are over the cells that are measured in the data.

In [5]:
real_path = os.path.join(FINAL, "pred_insample_real.csv")
sources = {
    "literature": os.path.join(DATA_OUT, "y_literature_inner.csv"),
    "stage0": os.path.join(OUT, "stage0", "pred_insample_predicted.csv"),
    "structural": os.path.join(DATA_OUT, "y_structural_fit.csv"),
    "final": os.path.join(FINAL, "pred_insample_predicted.csv"),
    "loocv": os.path.join(OUT, "loocv", "loocv_pred.csv"),
}
baseline_path = os.path.join(OUT, "loocv", "loocv_baseline.csv")
missing = [k for k, v in sources.items() if not os.path.exists(v)] + ([] if os.path.exists(baseline_path) else ["loocv baseline"])

if not os.path.exists(real_path) or missing:
    print("cannot build stage_metrics.csv yet; missing:", missing or ["final/pred_insample_real.csv"])
    stage_metrics = None
else:
    real = pd.read_csv(real_path, index_col=0)
    stages = {k: pd.read_csv(v, index_col=0).reindex(index=real.index, columns=real.columns) for k, v in sources.items()}
    baseline = pd.read_csv(baseline_path, index_col=0).reindex(index=real.index, columns=real.columns)
    reference = {c: float(real[c].mean()) for c in real.columns}

    table = []
    for name, pred in stages.items():
        m = pred_metrics(real, pred, met_ref=reference, flux_ref=reference)
        m["stage"] = name
        table.append(m)
    m = pred_metrics(real, baseline, met_ref=reference, flux_ref=reference)
    m["stage"] = "loocv_training_mean_baseline"
    table.append(m)
    stage_metrics = pd.DataFrame(table).set_index("stage")

    stage0_row = json.load(open(os.path.join(OUT, "stage0", "stage_row.json")))
    final_state = json.load(open(os.path.join(FINAL, "final_summary.json")))
    stage_metrics["obj"] = [float(bd_lit.loc["MEAN", "total"]) if bd_lit is not None else np.nan, stage0_row["obj"],
                            structural_row["obj"], final_state["obj"], np.nan, np.nan]
    os.makedirs(DATA_OUT, exist_ok=True)
    stage_metrics.to_csv(os.path.join(DATA_OUT, "stage_metrics.csv"))
    print("stage_metrics.csv written to", DATA_OUT)
    display(stage_metrics[["obj", "rmse_met", "r2_met_permean", "rmse_flux_li", "r2_flux_li_permean", "rmse_met_meanbaseline", "rmse_flux_li_meanbaseline"]])

stage_metrics.csv written to /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/results/flux_model/paper_figs_data


,obj,rmse_met,r2_met_permean,rmse_flux_li,r2_flux_li_permean,rmse_met_meanbaseline,rmse_flux_li_meanbaseline
stage,,,,,,,
literature,7771.698717,0.228400,-1.101217,3.743317,-25.508651,0.157566,0.727047
stage0,11.745483,0.120675,0.413445,0.411894,0.679044,0.157566,0.727047
structural,10.908666,0.113934,0.477139,0.359038,0.756132,0.157566,0.727047
final,10.908666,0.113934,0.477139,0.359038,0.756132,0.157566,0.727047
loocv,NaN,0.205771,-0.705481,1.404231,-2.730366,0.157566,0.727047
loocv_training_mean_baseline,NaN,0.165854,-0.107972,0.761669,-0.097506,0.157566,0.727047
